# Iteration 2 — Monash Unnamed Location Exploration

## Objective

Identify unnamed locations in the City of Monash from the Iteration 1
application-ready Vicmap dataset. These records will later be matched against
authoritative external sources to enrich their names.

This notebook performs exploration only. It does not overwrite the Iteration 1
dataset.

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
def find_project_root(start: Path) -> Path:
    """Find the repository root by looking for the data and pipeline folders."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "pipeline").is_dir():
            return candidate

    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vicmap"
    / "vicmap_app_ready.csv"
)

print("Input file:", INPUT_PATH.relative_to(PROJECT_ROOT).as_posix())
print("File exists:", INPUT_PATH.exists())

Input file: data/processed/vicmap/vicmap_app_ready.csv
File exists: True


In [4]:
places = pd.read_csv(INPUT_PATH)

print(f"Rows: {len(places):,}")
print(f"Columns: {places.shape[1]}")
display(places.head())

Rows: 3,237
Columns: 14


,place_id,display_name,place_name,name_source,activity_category,classification_confidence,lga_name,longitude,latitude,feature_type,feature_subtype,decision,source_dataset,source_record_id
0,vicmap_foi_1343107,Carlton Gardens Tennis Club,Carlton Gardens Tennis Club,vicmap_name_label,court,high,MELBOURNE,144.973255,-37.802165,sport facility,tennis court,include,vicmap_foi,1343107
1,vicmap_foi_1343108,Carlton Gardens Tennis Club,Carlton Gardens Tennis Club,vicmap_name_label,court,high,MELBOURNE,144.973485,-37.802477,sport facility,tennis court,include,vicmap_foi,1343108
2,vicmap_foi_1343350,Carlton Gardens Tennis Club,Carlton Gardens Tennis Club,vicmap_name_label,court,high,MELBOURNE,144.973414,-37.802146,sport facility,tennis court,include,vicmap_foi,1343350
3,vicmap_foi_1343409,Carlton Gardens Tennis Club,Carlton Gardens Tennis Club,vicmap_name_label,court,high,MELBOURNE,144.973326,-37.802498,sport facility,tennis court,include,vicmap_foi,1343409
4,vicmap_foi_1343063,Docklands Sports Court,Docklands Sports Court,vicmap_name_label,court,high,MELBOURNE,144.947630,-37.820050,sport facility,netball court,include,vicmap_foi,1343063


In [5]:
places.info()

<class 'pandas.DataFrame'>
RangeIndex: 3237 entries, 0 to 3236
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   place_id                   3237 non-null   str    
 1   display_name               3237 non-null   str    
 2   place_name                 3237 non-null   str    
 3   name_source                3237 non-null   str    
 4   activity_category          3237 non-null   str    
 5   classification_confidence  3237 non-null   str    
 6   lga_name                   3237 non-null   str    
 7   longitude                  3237 non-null   float64
 8   latitude                   3237 non-null   float64
 9   feature_type               3237 non-null   str    
 10  feature_subtype            3237 non-null   str    
 11  decision                   3237 non-null   str    
 12  source_dataset             3237 non-null   str    
 13  source_record_id           3237 non-null   int64  
dtypes: 

In [6]:
expected_columns = {
    "place_id",
    "display_name",
    "place_name",
    "name_source",
    "activity_category",
    "lga_name",
    "longitude",
    "latitude",
    "feature_type",
    "feature_subtype",
    "source_dataset",
    "source_record_id",
}

missing_columns = expected_columns.difference(places.columns)

assert not missing_columns, f"Missing expected columns: {sorted(missing_columns)}"
assert places["place_id"].notna().all(), "Some place_id values are missing."
assert places["place_id"].is_unique, "place_id is not unique."

print("Basic input checks passed.")

Basic input checks passed.


### Initial data check

The Iteration 1 dataset contains 3,237 locations and 14 columns. All columns are complete with no missing values. Location coordinates are stored as numeric values, while descriptive fields are stored as text. The required columns are present, and `place_id` is complete and unique. Therefore, the dataset passes the initial checks and is ready for unnamed-location exploration.

In [7]:
# Find the unnamed places
unnamed_places = places.loc[
    places["name_source"].eq("generated_from_subtype")
].copy()

print(f"Total unnamed locations: {len(unnamed_places):,}")

display(
    unnamed_places["lga_name"]
    .value_counts()
    .rename_axis("lga_name")
    .reset_index(name="unnamed_count")
)

Total unnamed locations: 703


,lga_name,unnamed_count
0,MONASH,565
1,MELTON,111
2,MELBOURNE,27


In [9]:
# verification
assert len(unnamed_places) == 703, (
    f"Expected 703 unnamed locations, found {len(unnamed_places)}."
)

print("Iteration 1 unnamed count confirmed.")

Iteration 1 unnamed count confirmed.


In [10]:
# Filter other places, keep MONASH
monash_unnamed = unnamed_places.loc[
    unnamed_places["lga_name"].eq("MONASH")
].copy()

print(f"Monash unnamed locations: {len(monash_unnamed):,}")

display(monash_unnamed.head())

Monash unnamed locations: 565


,place_id,display_name,place_name,name_source,activity_category,classification_confidence,lga_name,longitude,latitude,feature_type,feature_subtype,decision,source_dataset,source_record_id
2514,vicmap_foi_1010743,Unnamed Netball Court - Monash - 1010743,Unnamed Netball Court - Monash - 1010743,generated_from_subtype,court,high,MONASH,145.137259,-37.909905,sport facility,netball court,include,vicmap_foi,1010743
2515,vicmap_foi_1016170,Unnamed Netball Court - Monash - 1016170,Unnamed Netball Court - Monash - 1016170,generated_from_subtype,court,high,MONASH,145.099179,-37.906070,sport facility,netball court,include,vicmap_foi,1016170
2516,vicmap_foi_78115,Unnamed Netball Court - Monash - 78115,Unnamed Netball Court - Monash - 78115,generated_from_subtype,court,high,MONASH,145.106184,-37.864861,sport facility,netball court,include,vicmap_foi,78115
2517,vicmap_foi_78118,Unnamed Netball Court - Monash - 78118,Unnamed Netball Court - Monash - 78118,generated_from_subtype,court,high,MONASH,145.106527,-37.865357,sport facility,netball court,include,vicmap_foi,78118
2518,vicmap_foi_990846,Unnamed Netball Court - Monash - 990846,Unnamed Netball Court - Monash - 990846,generated_from_subtype,court,high,MONASH,145.193431,-37.895053,sport facility,netball court,include,vicmap_foi,990846


In [11]:
# verification
assert len(monash_unnamed) == 565
assert monash_unnamed["lga_name"].eq("MONASH").all()
assert monash_unnamed["name_source"].eq(
    "generated_from_subtype"
).all()

print("Monash unnamed subset checks passed.")

Monash unnamed subset checks passed.


### Unnamed location identification

The `name_source` field identifies 703 locations whose names were generated
from their feature subtype during Iteration 1. Of these, 565 are located in
Monash, 111 in Melton and 27 in Melbourne. The 565 Monash records form the
target dataset for the first stage of Iteration 2 name enrichment.

In [13]:
# checking the overview info about unnamed places
category_summary = (
    monash_unnamed
    .groupby(["activity_category", "feature_subtype"])
    .size()
    .reset_index(name="location_count")
    .sort_values(
        ["activity_category", "location_count"],
        ascending=[True, False],
    )
)

display(category_summary)

,activity_category,feature_subtype,location_count
1,court,tennis court,29
0,court,netball court,6
2,park_and_garden,park,289
3,playground,playground,142
8,sports_ground,sports ground,75
5,sports_ground,baseball field,16
7,sports_ground,sports complex,4
4,sports_ground,athletic field,2
6,sports_ground,hockey ground,2


In [14]:
quality_checks = pd.Series(
    {
        "rows": len(monash_unnamed),
        "missing_place_id": monash_unnamed["place_id"].isna().sum(),
        "duplicate_place_id": monash_unnamed["place_id"].duplicated().sum(),
        "missing_longitude": monash_unnamed["longitude"].isna().sum(),
        "missing_latitude": monash_unnamed["latitude"].isna().sum(),
        "duplicate_coordinates": monash_unnamed.duplicated(
            subset=["longitude", "latitude"]
        ).sum(),
    },
    name="count",
)

display(quality_checks.to_frame())

,count
rows,565
missing_place_id,0
duplicate_place_id,0
missing_longitude,0
missing_latitude,0
duplicate_coordinates,0


In [15]:
assert monash_unnamed["place_id"].notna().all()
assert monash_unnamed["place_id"].is_unique
assert monash_unnamed[["longitude", "latitude"]].notna().all().all()

print("Monash target data quality checks passed.")

Monash target data quality checks passed.


In [16]:
display(
    monash_unnamed[["longitude", "latitude"]]
    .agg(["min", "max"])
)

,longitude,latitude
min,145.075364,-37.936974
max,145.211606,-37.856231


### Monash target profile

The Monash target subset contains 565 records across four activity categories.
Parks and playgrounds account for most records. All target records have unique
place IDs and complete coordinates, with no exact coordinate duplicates.
The records are therefore suitable for spatial matching against an external
Monash Council dataset.

### Exploration of Monash Council data

In [19]:
from datetime import datetime, timezone

import pandas as pd
import requests

In [ ]:
# Define the official Monash Council source.


### Automated access result

The official Monash Parks and Recreation page returned HTTP 403 when accessed
through Python. This indicates that direct automated retrieval is not available.
No attempt was made to bypass the website's access controls. A downloadable
dataset, documented API or explicit permission is required before automated
collection can continue.

### Exploration of VPA Open Space

In [21]:
import geopandas as gpd

In [22]:
# Define the official VPA ArcGIS service.
VPA_API_URL = (
    "https://services5.arcgis.com/DmRfik4clMVydXO3/"
    "arcgis/rest/services/"
    "VPA_Draft_Open_Space_Data/FeatureServer/0/query"
)

# Request Monash polygons in WGS84 coordinates.
vpa_params = {
    "where": "LGA = 'MONASH'",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "resultRecordCount": 2000,
    "f": "geojson",
}

vpa_response = requests.get(
    VPA_API_URL,
    params=vpa_params,
    timeout=60,
)

vpa_response.raise_for_status()
vpa_data = vpa_response.json()

print("HTTP status:", vpa_response.status_code)
print("Response type:", vpa_data.get("type"))
print("Features returned:", len(vpa_data.get("features", [])))

HTTP status: 200
Response type: FeatureCollection
Features returned: 831


In [23]:
# Convert the GeoJSON response to a GeoDataFrame.
vpa_open_space = gpd.GeoDataFrame.from_features(
    vpa_data["features"],
    crs="EPSG:4326",
)

print("Rows:", len(vpa_open_space))
print("Columns:", len(vpa_open_space.columns))
print("CRS:", vpa_open_space.crs)

display(vpa_open_space.head())

Rows: 831
Columns: 23
CRS: EPSG:4326


,geometry,FID,LGA,VM_PARCEL_,VM_PARCE_1,DATA_SOURC,OS_CATEGOR,OS_CATEG_2,OWNER_TYPE,PARK_NAME,...,HA,SUBREGION,VEAC_ID,WATER_BODY,OS_TYPE,COASTAL,MANAGER_NA,OWNER_NAME,Image_URL,VPA_ID
0,"POLYGON ((145.157 -37.89854, 145.15698 -37.898...",8005,MONASH,RES1\PS340701,33571,ARCUE2002,Parks and gardens,Not applicable,Local government,"31 The Quadrangle, Glen Waverley",...,0.0889,Eastern,M013877,,Public open space,,NO DATA,Monash City Council,https://lh3.googleusercontent.com/-rdTUgqfNyXg...,9800
1,"POLYGON ((145.21043 -37.93661, 145.21046 -37.9...",8007,MONASH,RES1\PS335351,524467,ARCUE2002,Parks and gardens,Not applicable,Local government,Blanton Drive Reserve,...,0.3706,Eastern,M013879,,Public open space,,NO DATA,Monash City Council,https://lh3.googleusercontent.com/-rdTUgqfNyXg...,9801
2,"POLYGON ((145.19829 -37.93035, 145.19823 -37.9...",8008,MONASH,RES1\LP212901,2003664,ARCUE2002,Services and utilities reserves,Parks and gardens,Local government,Maygrove Way Reserve,...,0.7560,Eastern,M013880,,Restricted public land,,NO DATA,Monash City Council,https://lh3.googleusercontent.com/-0bRI1Ds9dCY...,9802
3,"POLYGON ((145.18761 -37.8839, 145.18757 -37.88...",8011,MONASH,RES2\LP137414,95922,ARCUE2002,Recreation corridor,Local link,Local government,Torwood Avenue Reserve,...,0.1158,Eastern,M013288,,Public open space,,NO DATA,Monash City Council,https://lh3.googleusercontent.com/-YzyOGz7u_dk...,9803
4,"POLYGON ((145.11451 -37.89579, 145.11449 -37.8...",8013,MONASH,RES1\PS412683,52420876,ARCUE2002,Transport reservations,Green buffer,Local government,Stanley Avenue Reserve,...,0.0443,Eastern,M013886,,Restricted public land,,NO DATA,Monash City Council,https://lh3.googleusercontent.com/-7jagzn1L1NA...,9804


In [24]:
# Remove surrounding whitespace from park names.
vpa_open_space["park_name_clean"] = (
    vpa_open_space["PARK_NAME"]
    .astype("string")
    .str.strip()
)

# Treat blank and placeholder values as missing.
invalid_name = (
    vpa_open_space["park_name_clean"].isna()
    | vpa_open_space["park_name_clean"].eq("")
    | vpa_open_space["park_name_clean"].str.upper().eq("NO DATA")
)

vpa_open_space.loc[invalid_name, "park_name_clean"] = pd.NA

In [25]:
# Summarise name availability.
vpa_name_summary = pd.Series(
    {
        "total_polygons": len(vpa_open_space),
        "named_polygons": vpa_open_space["park_name_clean"].notna().sum(),
        "unnamed_polygons": vpa_open_space["park_name_clean"].isna().sum(),
        "unique_names": vpa_open_space["park_name_clean"].nunique(),
    },
    name="count",
)

display(vpa_name_summary.to_frame())

,count
total_polygons,831
named_polygons,787
unnamed_polygons,44
unique_names,509


In [26]:
# Convert the Monash target records to spatial points.
monash_points = gpd.GeoDataFrame(
    monash_unnamed.copy(),
    geometry=gpd.points_from_xy(
        monash_unnamed["longitude"],
        monash_unnamed["latitude"],
    ),
    crs="EPSG:4326",
)

print("Monash target points:", len(monash_points))

Monash target points: 565


In [27]:
# Keep only named polygons and required matching fields.
named_vpa_polygons = vpa_open_space.loc[
    vpa_open_space["park_name_clean"].notna(),
    [
        "FID",
        "park_name_clean",
        "OS_CATEGOR",
        "OS_CATEG_2",
        "OS_STATUS",
        "OS_ACCESS",
        "geometry",
    ],
].copy()

# Match each target point to the polygon containing it.
vpa_matches = gpd.sjoin(
    monash_points,
    named_vpa_polygons,
    how="left",
    predicate="within",
)

print("Spatial join rows:", len(vpa_matches))

Spatial join rows: 565


In [28]:
# Keep one matched result per target location.
matched_locations = (
    vpa_matches.loc[vpa_matches["park_name_clean"].notna()]
    .drop_duplicates(subset="place_id")
    .copy()
)

# Count target and matched records by category.
target_count = (
    monash_points.groupby("activity_category")
    .size()
    .rename("target")
)

matched_count = (
    matched_locations.groupby("activity_category")
    .size()
    .rename("matched")
)

coverage_summary = (
    pd.concat([target_count, matched_count], axis=1)
    .fillna(0)
    .astype(int)
)

coverage_summary["unmatched"] = (
    coverage_summary["target"] - coverage_summary["matched"]
)

coverage_summary["coverage_percent"] = (
    coverage_summary["matched"]
    .div(coverage_summary["target"])
    .mul(100)
    .round(1)
)

display(coverage_summary)

,target,matched,unmatched,coverage_percent
activity_category,,,,
court,35,31,4,88.6
park_and_garden,289,222,67,76.8
playground,142,139,3,97.9
sports_ground,99,99,0,100.0


In [29]:
# Report overall potential coverage.
matched_total = matched_locations["place_id"].nunique()
target_total = monash_points["place_id"].nunique()

print(f"Matched: {matched_total:,}")
print(f"Unmatched: {target_total - matched_total:,}")
print(f"Potential coverage: {matched_total / target_total:.1%}")

Matched: 491
Unmatched: 74
Potential coverage: 86.9%


In [30]:
# Identify points contained by more than one named polygon.
candidate_counts = (
    vpa_matches.loc[vpa_matches["park_name_clean"].notna()]
    .groupby("place_id")
    .size()
)

multiple_candidate_count = candidate_counts.gt(1).sum()

print("Points with multiple polygon matches:", multiple_candidate_count)

Points with multiple polygon matches: 0


### Initial VPA matching result

The VPA Open Space dataset contains 831 Monash polygons, including 787
polygons with usable park or reserve names. Point-in-polygon matching produced
a single candidate name for 491 of the 565 Monash unnamed locations, giving
potential coverage of 86.9%. No target point matched more than one named
polygon. These results are candidates only and have not yet overwritten the
Iteration 1 names.

In [31]:
# Add access, ownership and source fields to matched candidates.
review_fields = [
    "OS_TYPE",
    "OWNER_TYPE",
    "MANAGER_TY",
    "MANAGER_NA",
    "OWNER_NAME",
    "DATA_SOURC",
]

candidate_matches = (
    vpa_matches.loc[vpa_matches["park_name_clean"].notna()]
    .join(
        vpa_open_space[review_fields],
        on="index_right",
        rsuffix="_vpa",
    )
    .drop_duplicates(subset="place_id")
    .copy()
)

print("Candidate matches:", len(candidate_matches))

Candidate matches: 491


In [32]:
# Compare target activity categories with VPA land categories.
category_comparison = pd.crosstab(
    candidate_matches["activity_category"],
    candidate_matches["OS_CATEGOR"],
)

display(category_comparison)

OS_CATEGOR,Government schools,Natural and semi-natural open space,Non-government schools,Parks and gardens,Recreation corridor,Services and utilities reserves,Sportsfields and organised recreation,Tertiary institutions,Transport reservations
activity_category,,,,,,,,,
court,4,0,3,0,0,0,21,3,0
park_and_garden,0,12,0,127,28,21,3,0,31
playground,6,14,1,87,1,0,27,0,3
sports_ground,12,2,18,0,0,0,60,7,0


In [37]:
# Summarise exploratory matching coverage.
matched_count = candidate_matches["place_id"].nunique()
unmatched_count = len(monash_unnamed) - matched_count
coverage_percent = matched_count / len(monash_unnamed) * 100

print(f"Monash unnamed locations: {len(monash_unnamed):,}")
print(f"VPA candidate matches: {matched_count:,}")
print(f"Unmatched locations: {unmatched_count:,}")
print(f"Potential coverage: {coverage_percent:.1f}%")

Monash unnamed locations: 565
VPA candidate matches: 491
Unmatched locations: 74
Potential coverage: 86.9%


In [38]:
# Confirm one candidate at most for each Vicmap location.
assert candidate_matches["place_id"].is_unique
assert candidate_matches["park_name_clean"].notna().all()
assert matched_count == 491
assert matched_count + unmatched_count == len(monash_unnamed)

print("Exploratory VPA matching checks passed.")

Exploratory VPA matching checks passed.


### Exploration of OpenStreetMap name coverage

This section retrieves named OpenStreetMap recreation features around Monash and
measures how many of the 565 target locations have a plausible nearby match.
No existing names are overwritten during this exploration.

In [39]:
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import LineString, Point, Polygon


# Build a bounding box around the 565 Monash target locations.
padding = 0.01

south = monash_unnamed["latitude"].min() - padding
north = monash_unnamed["latitude"].max() + padding
west = monash_unnamed["longitude"].min() - padding
east = monash_unnamed["longitude"].max() + padding

bbox = f"{south},{west},{north},{east}"


# Request named recreation features from OpenStreetMap.
osm_query = f"""
[out:json][timeout:180];
(
  nwr["name"]["leisure"~"park|garden|playground|pitch|sports_centre|stadium|recreation_ground|nature_reserve|track"]({bbox});
  nwr["name"]["sport"]({bbox});
  nwr["name"]["landuse"="recreation_ground"]({bbox});
  nwr["name"]["amenity"~"school|kindergarten|community_centre"]({bbox});
);
out tags center geom meta;
"""

overpass_urls = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

osm_response = None

# Try a backup server if the first Overpass server is unavailable.
for url in overpass_urls:
    try:
        response = requests.post(
            url,
            data={"data": osm_query},
            headers={
                "User-Agent": "active-together-student-project/iteration2"
            },
            timeout=240,
        )
        response.raise_for_status()
        osm_response = response
        break
    except requests.RequestException as error:
        print(f"Request failed: {url}")
        print(error)

if osm_response is None:
    raise RuntimeError("All Overpass API requests failed.")

osm_json = osm_response.json()
osm_retrieved_at = datetime.now(timezone.utc).isoformat()

print("OSM elements returned:", len(osm_json["elements"]))
print("Retrieved at:", osm_retrieved_at)

OSM elements returned: 661
Retrieved at: 2026-09-10T04:46:59.727520+00:00


In [40]:
def osm_element_geometry(element):
    """Convert an OSM element into a Shapely geometry."""

    # Nodes already contain one coordinate.
    if element["type"] == "node":
        return Point(element["lon"], element["lat"])

    coordinates = [
        (item["lon"], item["lat"])
        for item in element.get("geometry", [])
    ]

    # Closed OSM ways are treated as polygons.
    if len(coordinates) >= 4 and coordinates[0] == coordinates[-1]:
        geometry = Polygon(coordinates)
        return geometry if geometry.is_valid else geometry.buffer(0)

    # Open ways are treated as lines.
    if len(coordinates) >= 2:
        return LineString(coordinates)

    # Use the supplied centre for relations without simple geometry.
    centre = element.get("center")
    if centre:
        return Point(centre["lon"], centre["lat"])

    return None


osm_rows = []

# Preserve names, feature tags and provenance for later validation.
for element in osm_json["elements"]:
    tags = element.get("tags", {})
    geometry = osm_element_geometry(element)

    if geometry is None:
        continue

    osm_rows.append(
        {
            "osm_type": element["type"],
            "osm_id": element["id"],
            "osm_name": tags.get("name"),
            "osm_leisure": tags.get("leisure"),
            "osm_sport": tags.get("sport"),
            "osm_landuse": tags.get("landuse"),
            "osm_amenity": tags.get("amenity"),
            "osm_timestamp": element.get("timestamp"),
            "osm_check_date": tags.get("check_date"),
            "osm_source": tags.get("source"),
            "geometry": geometry,
        }
    )

osm_features = gpd.GeoDataFrame(
    osm_rows,
    geometry="geometry",
    crs="EPSG:4326",
).drop_duplicates(subset=["osm_type", "osm_id"])

print("Named OSM recreation features:", len(osm_features))
display(osm_features.head())

Named OSM recreation features: 649


,osm_type,osm_id,osm_name,osm_leisure,osm_sport,osm_landuse,osm_amenity,osm_timestamp,osm_check_date,osm_source,geometry
0,node,148544339,Syndal Pre-School,NaN,NaN,NaN,kindergarten,2022-07-07T22:33:30Z,NaN,NaN,POINT (145.14878 -37.8742)
1,node,191834621,Tally Ho Preschool,NaN,NaN,NaN,kindergarten,2022-07-07T22:33:30Z,NaN,NaN,POINT (145.16429 -37.86911)
2,node,207718805,St Johns Pre-School,NaN,NaN,NaN,kindergarten,2022-07-07T22:33:30Z,NaN,NaN,POINT (145.11374 -37.898)
3,node,246969693,Waverley Foothills Preschool,NaN,NaN,NaN,kindergarten,2022-07-11T08:01:18Z,NaN,NaN,POINT (145.2001 -37.93164)
4,node,257850907,Hughesdale Kindergarten,NaN,NaN,NaN,kindergarten,2018-01-24T03:21:45Z,NaN,NaN,POINT (145.07789 -37.8971)


In [41]:
# Use a projected CRS so distances are measured in metres.
target_projected = monash_points.to_crs("EPSG:7855")
osm_projected = osm_features.to_crs("EPSG:7855")

candidate_groups = []


def compatible_osm_features(activity_category):
    """Select plausible OSM feature types for each Vicmap category."""

    leisure = osm_projected["osm_leisure"]
    sport = osm_projected["osm_sport"].notna()
    landuse = osm_projected["osm_landuse"]
    amenity = osm_projected["osm_amenity"]

    park_context = (
        leisure.isin(
            [
                "park",
                "garden",
                "nature_reserve",
                "recreation_ground",
            ]
        )
        | landuse.eq("recreation_ground")
    )

    school_context = amenity.isin(
        ["school", "kindergarten", "community_centre"]
    )

    if activity_category == "court":
        return osm_projected.loc[
            leisure.isin(["pitch", "sports_centre", "stadium"])
            | sport
            | park_context
            | school_context
        ]

    if activity_category == "playground":
        return osm_projected.loc[
            leisure.eq("playground")
            | park_context
            | school_context
        ]

    if activity_category == "park_and_garden":
        return osm_projected.loc[park_context]

    if activity_category == "sports_ground":
        return osm_projected.loc[
            leisure.isin(
                [
                    "pitch",
                    "sports_centre",
                    "stadium",
                    "track",
                    "recreation_ground",
                ]
            )
            | sport
            | park_context
            | school_context
        ]

    return osm_projected.iloc[0:0]


# Find the nearest compatible OSM feature for each activity category.
for category in target_projected["activity_category"].unique():
    targets = target_projected.loc[
        target_projected["activity_category"].eq(category)
    ].copy()

    candidates = compatible_osm_features(category).copy()

    if candidates.empty:
        continue

    nearest = gpd.sjoin_nearest(
        targets,
        candidates,
        how="left",
        distance_col="osm_distance_m",
    )

    candidate_groups.append(nearest)

osm_nearest = pd.concat(candidate_groups, ignore_index=True)

# Keep one nearest candidate when several features are equally close.
best_osm_candidates = (
    osm_nearest
    .sort_values(["place_id", "osm_distance_m", "osm_id"])
    .drop_duplicates(subset="place_id")
    .copy()
)

# Show coverage under several exploratory distance limits.
coverage_summary = pd.DataFrame(
    {
        "maximum_distance_m": [50, 100, 200],
        "matched_locations": [
            best_osm_candidates.loc[
                best_osm_candidates["osm_name"].notna()
                & best_osm_candidates["osm_distance_m"].le(distance),
                "place_id",
            ].nunique()
            for distance in [50, 100, 200]
        ],
    }
)

coverage_summary["unmatched_locations"] = (
    len(monash_unnamed) - coverage_summary["matched_locations"]
)

coverage_summary["coverage_percent"] = (
    coverage_summary["matched_locations"]
    / len(monash_unnamed)
    * 100
).round(1)

display(coverage_summary)

,maximum_distance_m,matched_locations,unmatched_locations,coverage_percent
0,50,233,332,41.2
1,100,244,321,43.2
2,200,290,275,51.3


In [42]:
# Use 50 metres as the conservative exploration threshold.
OSM_DISTANCE_LIMIT_M = 50

osm_candidates_50 = best_osm_candidates.loc[
    best_osm_candidates["osm_name"].notna()
    & best_osm_candidates["osm_distance_m"].le(OSM_DISTANCE_LIMIT_M)
].copy()


def clean_osm_tag(value):
    """Convert a missing OSM tag to an empty string."""
    return "" if pd.isna(value) else str(value).strip()


def classify_osm_match(row):
    """Classify whether the OSM name describes the facility itself."""

    category = clean_osm_tag(row["activity_category"])
    leisure = clean_osm_tag(row["osm_leisure"])
    sport = clean_osm_tag(row["osm_sport"])
    landuse = clean_osm_tag(row["osm_landuse"])
    amenity = clean_osm_tag(row["osm_amenity"])

    # Courts should match a named pitch or another sport facility.
    if category == "court":
        if leisure in {"pitch", "sports_centre", "stadium"} or sport:
            return "direct_feature"
        return "context_name"

    # A playground tag represents the facility directly.
    if category == "playground":
        if leisure == "playground":
            return "direct_feature"
        return "context_name"

    # Named parks and recreation grounds directly support this category.
    if category == "park_and_garden":
        if (
            leisure
            in {
                "park",
                "garden",
                "nature_reserve",
                "recreation_ground",
            }
            or landuse == "recreation_ground"
        ):
            return "direct_feature"
        return "context_name"

    # Named sport facilities directly support sports-ground records.
    if category == "sports_ground":
        if (
            leisure
            in {
                "pitch",
                "sports_centre",
                "stadium",
                "track",
                "recreation_ground",
            }
            or landuse == "recreation_ground"
            or sport
        ):
            return "direct_feature"
        return "context_name"

    return "review"


osm_candidates_50["osm_match_level"] = (
    osm_candidates_50.apply(classify_osm_match, axis=1)
)

print("OSM candidates within 50 m:", len(osm_candidates_50))

display(
    pd.crosstab(
        osm_candidates_50["activity_category"],
        osm_candidates_50["osm_match_level"],
        margins=True,
    )
)

OSM candidates within 50 m: 233


osm_match_level,context_name,direct_feature,All
activity_category,,,
court,19,6,25
park_and_garden,0,32,32
playground,81,2,83
sports_ground,68,25,93
All,168,65,233


In [43]:
# Summarise candidate distances by match level.
distance_summary = (
    osm_candidates_50
    .groupby("osm_match_level")["osm_distance_m"]
    .agg(["count", "min", "median", "max"])
    .round(1)
)

display(distance_summary)


# Inspect the OSM feature types supplying candidate names.
tag_summary = (
    osm_candidates_50
    .groupby(
        [
            "osm_match_level",
            "osm_leisure",
            "osm_amenity",
            "osm_landuse",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="locations")
    .sort_values("locations", ascending=False)
)

display(tag_summary.head(20))

,count,min,median,max
osm_match_level,,,,
context_name,168,0.0,0.0,49.1
direct_feature,65,0.0,0.0,47.7


,osm_match_level,osm_leisure,osm_amenity,osm_landuse,locations
1,context_name,park,NaN,NaN,120
3,context_name,NaN,school,NaN,41
7,direct_feature,park,NaN,NaN,24
11,direct_feature,sports_centre,NaN,NaN,12
9,direct_feature,pitch,NaN,NaN,10
12,direct_feature,NaN,NaN,recreation_ground,8
5,direct_feature,park,NaN,pipe_line,5
2,context_name,NaN,kindergarten,NaN,3
4,context_name,NaN,NaN,recreation_ground,3
8,direct_feature,pitch,NaN,recreation_ground,2


In [44]:
# Retain all equally near candidates before selecting one per place.
raw_candidates_50 = osm_nearest.loc[
    osm_nearest["osm_name"].notna()
    & osm_nearest["osm_distance_m"].le(OSM_DISTANCE_LIMIT_M)
].copy()

ambiguity_summary = (
    raw_candidates_50
    .groupby("place_id")
    .agg(
        candidate_records=("osm_id", "size"),
        distinct_osm_names=("osm_name", "nunique"),
    )
    .reset_index()
)

ambiguous_place_ids = ambiguity_summary.loc[
    ambiguity_summary["distinct_osm_names"].gt(1),
    "place_id",
]

print("Locations with multiple distinct nearest names:",
      len(ambiguous_place_ids))

ambiguous_candidates = (
    raw_candidates_50.loc[
        raw_candidates_50["place_id"].isin(ambiguous_place_ids),
        [
            "place_id",
            "activity_category",
            "osm_name",
            "osm_leisure",
            "osm_sport",
            "osm_amenity",
            "osm_distance_m",
            "osm_type",
            "osm_id",
        ],
    ]
    .sort_values(["place_id", "osm_distance_m", "osm_name"])
)

display(ambiguous_candidates.head(20))

Locations with multiple distinct nearest names: 12


,place_id,activity_category,osm_name,osm_leisure,osm_sport,osm_amenity,osm_distance_m,osm_type,osm_id
509,vicmap_foi_1166416,sports_ground,Wesley College Glen Waverley Campus,NaN,NaN,school,0.0,way,159949778
510,vicmap_foi_1166416,sports_ground,Williams Oval,pitch,cricket,NaN,0.0,way,50319075
15,vicmap_foi_990812,court,Capital Reserve,park,NaN,NaN,0.0,way,22724348
14,vicmap_foi_990812,court,Legend Park Tennis Club,sports_centre,tennis,NaN,0.0,way,1550771284
533,vicmap_foi_990818,sports_ground,Argyle Reserve,park,NaN,NaN,0.0,way,57772409
534,vicmap_foi_990818,sports_ground,Monash City Football Club,pitch,soccer,NaN,0.0,way,1270603094
541,vicmap_foi_990837,sports_ground,Jack Meade Oval,pitch,cricket,NaN,0.0,way,122212716
542,vicmap_foi_990837,sports_ground,Jack Meade Reserve,park,NaN,NaN,0.0,way,24237884
545,vicmap_foi_990840,sports_ground,Athletics Track,track,NaN,NaN,0.0,way,365963502
543,vicmap_foi_990840,sports_ground,Huntingtower School,NaN,NaN,school,0.0,way,23053503


In [45]:
# Confirm that this remains an independent exploration result.
assert len(monash_unnamed) == 565
assert osm_candidates_50["place_id"].is_unique
assert osm_candidates_50["osm_name"].notna().all()
assert osm_candidates_50["osm_distance_m"].le(50).all()
assert set(osm_candidates_50["osm_match_level"]).issubset(
    {"direct_feature", "context_name", "review"}
)

print("OSM exploratory candidate checks passed.")

OSM exploratory candidate checks passed.


### VPA VS OSM

In [46]:
import re
import unicodedata


def normalize_place_name(value):
    """Normalise names for conservative equality comparison."""

    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9]+", " ", text.lower())

    return " ".join(text.split())


# Keep one VPA candidate per Vicmap location.
vpa_name_candidates = (
    candidate_matches[
        [
            "place_id",
            "park_name_clean",
            "OS_CATEGOR",
            "OS_CATEG_2",
        ]
    ]
    .drop_duplicates(subset="place_id")
    .rename(columns={"park_name_clean": "vpa_name"})
)

# Keep the conservative 50-metre OSM candidates.
osm_name_candidates = osm_candidates_50[
    [
        "place_id",
        "osm_name",
        "osm_match_level",
        "osm_distance_m",
        "osm_type",
        "osm_id",
        "osm_leisure",
        "osm_sport",
        "osm_landuse",
        "osm_amenity",
        "osm_timestamp",
        "osm_check_date",
        "osm_source",
    ]
].copy()

print("VPA candidates:", len(vpa_name_candidates))
print("OSM candidates within 50 m:", len(osm_name_candidates))

VPA candidates: 491
OSM candidates within 50 m: 233


In [47]:
# Join both sources without modifying the original target data.
source_comparison = (
    monash_unnamed[
        [
            "place_id",
            "display_name",
            "activity_category",
            "feature_subtype",
            "longitude",
            "latitude",
        ]
    ]
    .merge(
        vpa_name_candidates,
        on="place_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        osm_name_candidates,
        on="place_id",
        how="left",
        validate="one_to_one",
    )
)

source_comparison["has_vpa_name"] = (
    source_comparison["vpa_name"].notna()
)

source_comparison["has_osm_name"] = (
    source_comparison["osm_name"].notna()
)


def classify_source_coverage(row):
    """Describe which external sources matched each location."""

    if row["has_vpa_name"] and row["has_osm_name"]:
        return "both_sources"

    if row["has_vpa_name"]:
        return "vpa_only"

    if row["has_osm_name"]:
        return "osm_only"

    return "neither_source"


source_comparison["source_coverage"] = source_comparison.apply(
    classify_source_coverage,
    axis=1,
)

coverage_order = [
    "both_sources",
    "vpa_only",
    "osm_only",
    "neither_source",
]

coverage_summary = (
    source_comparison["source_coverage"]
    .value_counts()
    .reindex(coverage_order, fill_value=0)
    .rename_axis("source_coverage")
    .reset_index(name="locations")
)

coverage_summary["percentage"] = (
    coverage_summary["locations"]
    / len(source_comparison)
    * 100
).round(1)

display(coverage_summary)

,source_coverage,locations,percentage
0,both_sources,226,40.0
1,vpa_only,265,46.9
2,osm_only,7,1.2
3,neither_source,67,11.9


In [48]:
# Compare names only where both sources produced a candidate.
source_comparison["vpa_name_normalized"] = (
    source_comparison["vpa_name"].apply(normalize_place_name)
)

source_comparison["osm_name_normalized"] = (
    source_comparison["osm_name"].apply(normalize_place_name)
)

both_mask = (
    source_comparison["has_vpa_name"]
    & source_comparison["has_osm_name"]
)

same_name_mask = (
    both_mask
    & source_comparison["vpa_name_normalized"].eq(
        source_comparison["osm_name_normalized"]
    )
)

source_comparison["name_comparison"] = "not_applicable"
source_comparison.loc[both_mask, "name_comparison"] = (
    "different_name"
)
source_comparison.loc[same_name_mask, "name_comparison"] = (
    "same_normalized_name"
)

name_comparison_summary = (
    source_comparison.loc[both_mask, "name_comparison"]
    .value_counts()
    .rename_axis("name_comparison")
    .reset_index(name="locations")
)

display(name_comparison_summary)

,name_comparison,locations
0,same_normalized_name,120
1,different_name,106


In [49]:
# Separate direct OSM names from contextual park or school names.
osm_only_summary = pd.crosstab(
    source_comparison.loc[
        source_comparison["source_coverage"].eq("osm_only"),
        "activity_category",
    ],
    source_comparison.loc[
        source_comparison["source_coverage"].eq("osm_only"),
        "osm_match_level",
    ],
    margins=True,
)

display(osm_only_summary)


# Inspect the strongest incremental OSM candidates first.
osm_only_direct = (
    source_comparison.loc[
        source_comparison["source_coverage"].eq("osm_only")
        & source_comparison["osm_match_level"].eq("direct_feature"),
        [
            "place_id",
            "activity_category",
            "feature_subtype",
            "osm_name",
            "osm_distance_m",
            "osm_leisure",
            "osm_sport",
            "osm_timestamp",
            "longitude",
            "latitude",
        ],
    ]
    .sort_values(["osm_distance_m", "activity_category"])
)

print("OSM-only direct candidates:", len(osm_only_direct))
display(osm_only_direct.head(20))

osm_match_level,context_name,direct_feature,All
activity_category,,,
park_and_garden,0,5,5
playground,2,0,2
All,2,5,7


OSM-only direct candidates: 5


,place_id,activity_category,feature_subtype,osm_name,osm_distance_m,osm_leisure,osm_sport,osm_timestamp,longitude,latitude
51,vicmap_foi_1277258,park_and_garden,park,Scotchman's Creek Linear Park,0.000000,park,NaN,2026-05-27T22:52:57Z,145.135522,-37.886128
52,vicmap_foi_1333343,park_and_garden,park,Holmesglen Reserve,30.465779,park,NaN,2022-06-25T02:21:57Z,145.094717,-37.873265
69,vicmap_foi_643446,park_and_garden,park,Valley Conservation Reserve,37.606675,park,NaN,2025-10-22T07:21:56Z,145.133128,-37.877182
143,vicmap_foi_647898,park_and_garden,park,Glen Waverley North Reserve,46.511062,park,NaN,2025-11-10T20:42:53Z,145.159045,-37.868870
232,vicmap_foi_78292,park_and_garden,park,Hinkler Reserve,47.721812,park,NaN,2025-12-19T05:19:30Z,145.173876,-37.882215


In [50]:
# List records where both sources matched but supplied different names.
name_conflicts = (
    source_comparison.loc[
        source_comparison["name_comparison"].eq("different_name"),
        [
            "place_id",
            "activity_category",
            "vpa_name",
            "osm_name",
            "osm_match_level",
            "osm_distance_m",
            "osm_timestamp",
            "longitude",
            "latitude",
        ],
    ]
    .sort_values(
        ["osm_match_level", "osm_distance_m"],
        ascending=[True, True],
    )
)

print("Different VPA and OSM names:", len(name_conflicts))
display(name_conflicts.head(20))

Different VPA and OSM names: 106


,place_id,activity_category,vpa_name,osm_name,osm_match_level,osm_distance_m,osm_timestamp,longitude,latitude
2,vicmap_foi_78115,court,Ashwood Secondary College and Parkwood Primary...,Ashwood High School,context_name,0.0,2025-07-15T21:37:28Z,145.106184,-37.864861
3,vicmap_foi_78118,court,Ashwood Secondary College and Parkwood Primary...,Ashwood High School,context_name,0.0,2025-07-15T21:37:28Z,145.106527,-37.865357
9,vicmap_foi_78102,court,Ashwood Secondary College and Parkwood Primary...,Ashwood High School,context_name,0.0,2025-07-15T21:37:28Z,145.105052,-37.864520
325,vicmap_foi_1239045,playground,Brentwood Secondary College,Glen Waverley South Primary School,context_name,0.0,2025-01-07T07:30:20Z,145.167532,-37.897654
327,vicmap_foi_1239077,playground,Brentwood Secondary College,Glen Waverley South Primary School,context_name,0.0,2025-01-07T07:30:20Z,145.167876,-37.897469
328,vicmap_foi_1239091,playground,Brentwood Secondary College,Glen Waverley South Primary School,context_name,0.0,2025-01-07T07:30:20Z,145.168124,-37.897498
330,vicmap_foi_1239100,playground,Brentwood Secondary College,Glen Waverley South Primary School,context_name,0.0,2025-01-07T07:30:20Z,145.167641,-37.897456
342,vicmap_foi_987760,playground,Portland Street North Reserve,Portland Reserve,context_name,0.0,2019-12-09T22:10:17Z,145.205406,-37.935707
346,vicmap_foi_987846,playground,Danien Street Reserve,Danien Playground,context_name,0.0,2009-12-22T22:18:38Z,145.172452,-37.875862
347,vicmap_foi_987847,playground,View Point Avenue Reserve,Viewpoint Avenue Reserve,context_name,0.0,2025-02-17T09:02:22Z,145.171985,-37.889424


In [51]:
# Confirm that every target belongs to exactly one coverage group.
assert len(source_comparison) == 565
assert source_comparison["place_id"].is_unique
assert coverage_summary["locations"].sum() == 565

# Confirm that source flags agree with the coverage labels.
assert (
    source_comparison.loc[
        source_comparison["source_coverage"].eq("both_sources"),
        ["has_vpa_name", "has_osm_name"],
    ]
    .all()
    .all()
)

print("VPA and OSM exploratory comparison checks passed.")

VPA and OSM exploratory comparison checks passed.


In [52]:
# Add geometry type to help interpret zero and non-zero distances.
osm_geometry_types = osm_features[
    ["osm_type", "osm_id", "geometry"]
].copy()

osm_geometry_types["osm_geometry_type"] = (
    osm_geometry_types.geometry.geom_type
)

osm_geometry_types = osm_geometry_types.drop(
    columns="geometry"
)


# Isolate the seven records covered only by OSM.
osm_only_review = (
    source_comparison.loc[
        source_comparison["source_coverage"].eq("osm_only")
    ]
    .merge(
        osm_geometry_types,
        on=["osm_type", "osm_id"],
        how="left",
        validate="many_to_one",
    )
    .copy()
)


# Create a direct link to each OSM object.
osm_only_review["osm_url"] = osm_only_review.apply(
    lambda row: (
        f"https://www.openstreetmap.org/"
        f"{row['osm_type']}/{int(row['osm_id'])}"
    ),
    axis=1,
)


def assign_review_status(row):
    """Assign a conservative status before manual verification."""

    if row["osm_match_level"] == "context_name":
        return "context_only"

    if (
        row["osm_geometry_type"] in {"Polygon", "MultiPolygon"}
        and row["osm_distance_m"] == 0
    ):
        return "strong_spatial_candidate"

    return "manual_map_review"


osm_only_review["review_status"] = osm_only_review.apply(
    assign_review_status,
    axis=1,
)


review_columns = [
    "place_id",
    "activity_category",
    "feature_subtype",
    "osm_name",
    "osm_match_level",
    "osm_geometry_type",
    "osm_distance_m",
    "osm_timestamp",
    "review_status",
    "osm_url",
]

display(
    osm_only_review[review_columns]
    .sort_values(["review_status", "osm_distance_m"])
)

,place_id,activity_category,feature_subtype,osm_name,osm_match_level,osm_geometry_type,osm_distance_m,osm_timestamp,review_status,osm_url
5,vicmap_foi_1332412,playground,playground,Clayton Community Space West,context_name,Polygon,0.000000,2025-04-03T01:56:02Z,context_only,https://www.openstreetmap.org/way/1317789598
6,vicmap_foi_991061,playground,playground,Golf Links Avenue Reserve,context_name,Polygon,0.000000,2024-02-16T01:52:14Z,context_only,https://www.openstreetmap.org/way/61642577
1,vicmap_foi_1333343,park_and_garden,park,Holmesglen Reserve,direct_feature,Polygon,30.465779,2022-06-25T02:21:57Z,manual_map_review,https://www.openstreetmap.org/way/66800051
2,vicmap_foi_643446,park_and_garden,park,Valley Conservation Reserve,direct_feature,Polygon,37.606675,2025-10-22T07:21:56Z,manual_map_review,https://www.openstreetmap.org/way/4536039
3,vicmap_foi_647898,park_and_garden,park,Glen Waverley North Reserve,direct_feature,Polygon,46.511062,2025-11-10T20:42:53Z,manual_map_review,https://www.openstreetmap.org/way/14963677
4,vicmap_foi_78292,park_and_garden,park,Hinkler Reserve,direct_feature,Polygon,47.721812,2025-12-19T05:19:30Z,manual_map_review,https://www.openstreetmap.org/way/22902372
0,vicmap_foi_1277258,park_and_garden,park,Scotchman's Creek Linear Park,direct_feature,Polygon,0.000000,2026-05-27T22:52:57Z,strong_spatial_candidate,https://www.openstreetmap.org/way/4721862


### OpenStreetMap exploration outcome

OpenStreetMap (OSM) was explored as a supplementary source for the 565 Monash locations that had generated names. Using a conservative 50-metre threshold, 233 locations had a nearby named OSM candidate. However, 226 of these locations were already covered by the VPA Open Space dataset, leaving only seven additional OSM-only candidates.

Of the seven additional candidates, five represented directly related park features and two represented contextual park or community-space names rather than the target playground itself. Only one candidate had strong spatial evidence, while the remaining candidates required manual interpretation. In addition, 106 records matched by both VPA and OSM had different normalised names, often because the two sources described different spatial levels, such as a playground, school, reserve or wider park.

Given the limited additional coverage and the risk of assigning a nearby or parent-place name to a more specific Vicmap feature, OSM names were not used in the automated cleaning or replacement process. OSM was retained only as an exploratory and comparison source. The 74 locations not covered by VPA therefore retain their deterministic names generated during Iteration 1.

**Decision:** Use VPA candidate names for the 491 matched locations, subject to cleaning and validation. Do not apply OSM candidates to the final dataset.

**OSM source:** [OpenStreetMap](https://www.openstreetmap.org/copyright), accessed through the Overpass API. The retrieval timestamp is recorded in `osm_retrieved_at`.